# Disorder-averaged overlap on the Kagome torus

Run this notebook from the repository root, or from `notebook/`. The graph and the couplings come from `code/kagome_qea/`. The projector is `pqmc.project`: two replicas, a flat guide $\psi = 1$, and a forward-walked overlap. The one-copy energy notebook does not measure this.

The operator is

$$
H = -\sum_b J_b \sigma_i^z \sigma_j^z - \Gamma \sum_i \sigma_i^x - h \sum_i \sigma_i^z,
$$

with $s_i = \pm 1$, $J_b \sim N(0,1)$ on the nearest-neighbor bonds, and $h = 0$.


## The order parameter

At $h = 0$ the spin-flip symmetry makes every site magnetization average to zero. The glass is read from two replicas that share one draw of $J$ and do not interact. Their equal-time overlap is

$$
q = N^{-1} \sum_i s_i^{(a)} s_i^{(b)}.
$$

The walkers carry the ground-state amplitude, so a snapshot of $q$ is not $\langle q^n\rangle$. The overlap is tagged when a block is measured, descendants inherit the tag, and the tag is read after a further projection time $\tau_{\mathrm{forward}}$. That forward-walked average is the pure estimator. The Edwards–Anderson signal is the disorder average of the second moment,

$$
q_{\mathrm{EA}}^{2} = \bigl[\langle q^{2}\rangle\bigr].
$$

The Binder ratio is formed on each sample and then averaged,

$$
B = \frac{1}{2}\left(3 - \frac{\langle q^{4}\rangle}{\langle q^{2}\rangle^{2}}\right).
$$

$B$ is near 1 in a glass and near 0 in a paramagnet. A crossing of $B(\Gamma)$ between two sizes locates the finite-size transition. One falling curve of $q_{\mathrm{EA}}^{2}$ is only a hint. In the paramagnet $q_{\mathrm{EA}}^{2}$ sits near $1/N$.

Each of the 2500 walkers starts as an independent product state. The array for one walker has shape `(2, N)`: two replicas, and every entry is $+1$ or $-1$ with equal probability. The first 40 blocks, a projection time $\tau = 8$, are discarded.


## What this run is

`L = 4` is the $4\times 4$ torus, 48 sites. `N_SAMPLES = 32` is the number of independent draws of $J$, `sample_id = 0, ..., 31`, with entropy `[1000, L, sample_id, 1]`. It is not the number of transverse fields. The $\Gamma$ grid has seven values, so this size is $32 \times 7 = 224$ projector runs. The same notebook runs the $6\times 6$ torus by setting `L = 6` after this curve has fallen outside its disorder error. The $8\times 8$ torus is not part of this pass.

Eight forward-walking cycles at 2500 walkers missed the exact $L = 2$ overlap by about $0.10$, and each cycle kept only a few ancestral lines. This run uses 5000 walkers and 24 cycles of $\tau_{\mathrm{forward}} = 4$, which brings that twelve-site sample inside the check below. A draw at $L = 4$ then takes about an hour, and 32 draws take about a day and a half. A finished pair is reloaded, so the cell can be stopped and started again.

The plotted error is the standard deviation of the 32 sample values divided by $\sqrt{32}$. Twenty-four overlap values are still shorter than the length check in `series_error`, so a single sample is not marked resolved.

```text
data/kagome/gaussian_nn/pbc/L{L}/pqmc/h0/gamma{gamma}/sample{id}_seed0.npz
```


In [ ]:
from pathlib import Path
import sys
import time

import numpy as np

root = Path.cwd()
if not (root / "code").exists():
    root = root.parent
sys.path.insert(0, str(root / "code"))

from kagome_qea import make_sample
from kagome_qea.exact import ground_state
from kagome_qea.observables import jackknife_disorder_binder, jackknife_mean_error
from kagome_qea.pqmc import project

L = 4
N_SAMPLES = 32
GAMMAS = (0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5)
FIELD = 0.0
PROJECTOR_SEED = 0
CHECK_GAMMA = 1.0
overlap_checked = False
PROJECT = dict(
    n_replicas=2,
    n_walkers=5000,
    tau_block=0.2,
    n_equilibrate=40,
    n_measure=40,
    tau_forward=4.0,
    n_forward_cycles=24,
)

def token(value):
    text = f"{float(value):.8f}".rstrip("0").rstrip(".")
    if text in ("", "-0"):
        text = "0"
    return text.replace("-", "m").replace(".", "p")

def run_path(sample_id, gamma, size=None):
    directory = (
        root / "data" / "kagome" / "gaussian_nn" / "pbc"
        / f"L{L if size is None else size}" / "pqmc" / f"h{token(FIELD)}" / f"gamma{token(gamma)}"
    )
    return directory / f"sample{int(sample_id)}_seed{PROJECTOR_SEED}.npz"


In [ ]:
check_sample = make_sample(2, True, 0)
check_operator = check_sample.hamiltonian(FIELD)
exact = ground_state(check_operator, CHECK_GAMMA)
started = time.perf_counter()
check = project(check_operator, CHECK_GAMMA, seed=PROJECTOR_SEED, **PROJECT)
seconds = time.perf_counter() - started
difference = abs(check["q2"] - exact["q2"])
limit = max(0.06, 3.0 * check["q2_error"])
print(
    f"L=2 sample 0  Γ={CHECK_GAMMA:.1f}  {seconds:.1f}s\n"
    f"exact q2 {exact['q2']:.4f}  projector {check['q2']:.4f} ± {check['q2_error']:.4f}  "
    f"ancestors {check['min_unique_ancestors']}"
)
if check["q2_error"] > 0.06 or difference > limit:
    raise RuntimeError(
        f"L=2 overlap missed the exact value by {difference:.4f}, limit {limit:.4f}"
    )
overlap_checked = True


## Scan

The next cell is the $L = 4$ disorder average. It refuses to start until the twelve-site check above has passed.


In [ ]:
if not overlap_checked:
    raise RuntimeError("the L=2 overlap check has not passed")

def load_row(path):
    with np.load(path, allow_pickle=False) as data:
        row = {}
        for key in data.files:
            value = data[key]
            row[key] = value.item() if value.shape == () else np.array(value)
        return row

def save_row(path, sample, gamma, result, seconds):
    path.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(
        path,
        L=np.int64(sample.L),
        periodic=np.bool_(sample.periodic),
        sample_id=np.int64(sample.sample_id),
        ensemble=np.array(sample.ensemble),
        seed_entropy=np.asarray(sample.seed_entropy, dtype=np.int64),
        gamma=np.float64(gamma),
        h=np.float64(FIELD),
        seed=np.int64(PROJECTOR_SEED),
        n_sites=np.int64(sample.n_sites),
        seconds=np.float64(seconds),
        energy=np.float64(result["energy"]),
        energy_error=np.float64(result["energy_error"]),
        energy_growth=np.float64(result["energy_growth"]),
        energy_growth_error=np.float64(result["energy_growth_error"]),
        q=np.float64(result["q"]),
        q_error=np.float64(result["q_error"]),
        q_series=np.asarray(result["q_series"], dtype=np.float64),
        q2=np.float64(result["q2"]),
        q2_error=np.float64(result["q2_error"]),
        q2_series=np.asarray(result["q2_series"], dtype=np.float64),
        q4=np.float64(result["q4"]),
        q4_error=np.float64(result["q4_error"]),
        q4_series=np.asarray(result["q4_series"], dtype=np.float64),
        binder=np.float64(result["binder"]),
        min_unique_ancestors=np.int64(result["min_unique_ancestors"]),
        J=np.asarray(sample.J, dtype=np.float64),
    )

rows = []
for sample_id in range(N_SAMPLES):
    sample = make_sample(L, True, sample_id)
    operator = sample.hamiltonian(FIELD)
    for gamma in GAMMAS:
        path = run_path(sample_id, gamma)
        if path.exists():
            row = load_row(path)
            print(f"loaded  sample {sample_id:2d}  Γ={gamma:.1f}", flush=True)
        else:
            started = time.perf_counter()
            result = project(operator, gamma, seed=PROJECTOR_SEED, **PROJECT)
            seconds = time.perf_counter() - started
            save_row(path, sample, gamma, result, seconds)
            row = load_row(path)
            print(
                f"ran     sample {sample_id:2d}  Γ={gamma:.1f}  {seconds:7.1f}s  "
                f"q2 {result['q2']:.4f} ± {result['q2_error']:.4f}",
                flush=True,
            )
        rows.append(row)

n_sites = int(rows[0]["n_sites"])
averages = []
print(
    f"{'Γ':>5} {'[q2]':>8} {'err':>8} {'[B]':>8} {'err':>8} {'draws':>6}"
)
for gamma in GAMMAS:
    subset = [row for row in rows if float(row["gamma"]) == float(gamma)]
    second = np.array([row["q2"] for row in subset], dtype=float)
    fourth = np.array([row["q4"] for row in subset], dtype=float)
    binder, binder_error = jackknife_disorder_binder(second, fourth)
    averages.append({
        "gamma": float(gamma),
        "q2": float(second.mean()),
        "q2_error": jackknife_mean_error(second),
        "binder": binder,
        "binder_error": binder_error,
        "n_draws": int(second.size),
    })
    print(
        f"{gamma:5.1f} {second.mean():8.4f} {averages[-1]['q2_error']:8.4f} "
        f"{binder:8.4f} {binder_error:8.4f} {second.size:6d}"
    )


In [ ]:
import matplotlib.pyplot as plt

gammas = np.array([row["gamma"] for row in averages], dtype=float)
figure, axes = plt.subplots(1, 2, figsize=(9.0, 4.0))
axes[0].errorbar(
    gammas,
    [row["q2"] for row in averages],
    yerr=[row["q2_error"] for row in averages],
    fmt="o",
    capsize=3,
)
axes[0].axhline(1.0 / n_sites, color="0.45", linestyle="--", label="$1/N$")
axes[0].set_xlabel(r"$\Gamma$")
axes[0].set_ylabel(r"$[\langle q^{2}\rangle]$")
axes[0].legend()
axes[1].errorbar(
    gammas,
    [row["binder"] for row in averages],
    yerr=[row["binder_error"] for row in averages],
    fmt="o",
    capsize=3,
)
axes[1].set_xlabel(r"$\Gamma$")
axes[1].set_ylabel("$[B]$")
figure.suptitle(rf"${L}\times {L}$ Kagome torus, {N_SAMPLES} coupling draws, $h = 0$")
figure.tight_layout()
plt.show()
